# Build `llama.cpp` (CUDA) sekali, simpan sebagai Kaggle Dataset

Notebook ini **cuma dijalankan sekali** (atau setiap kali mau update ke versi `llama.cpp`
terbaru). Hasil build (`llama-server`, `llama-cli`, dan shared library-nya) disimpan sebagai
Kaggle Dataset, supaya notebook deploy DeepAgents kamu **tidak perlu compile dari source lagi**
tiap sesi baru — tinggal attach dataset ini dan langsung pakai binary-nya.

**Sebelum run:**
- Notebook Settings → Accelerator → **GPU T4 x2** (build perlu nvcc/CUDA toolkit yang aktif
  di environment, walau proses compile-nya sendiri CPU-bound).
- Internet → **On** (untuk clone repo).
- Kalau mau otomatis push ke Kaggle Dataset, siapkan Kaggle Secrets `KAGGLE_USERNAME` dan
  `KAGGLE_KEY` (lihat cell terakhir).


## 0. Cek versi CUDA di environment Kaggle

In [1]:
!nvcc --version
!nvidia-smi --query-gpu=index,name,compute_cap --format=csv

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
index, name, compute_cap
0, Tesla T4, 7.5
1, Tesla T4, 7.5


## 1. Patch, Clone & build `llama.cpp` dengan CUDA (sm75, untuk T4)

Sama seperti build manual sebelumnya, tapi kali ini hasilnya akan kita simpan permanen
sebagai dataset, jadi cukup dijalankan sekali.


In [2]:
# Clone repo llama.cpp (kalau folder sudah ada dari run sebelumnya, di-skip dan tinggal update)
%cd /kaggle/working

import os
if not os.path.isdir("/kaggle/working/llama.cpp"):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp

%cd /kaggle/working/llama.cpp

LLAMA_CPP_COMMIT = !git rev-parse --short HEAD
LLAMA_CPP_COMMIT = LLAMA_CPP_COMMIT[0]
print("Building commit:", LLAMA_CPP_COMMIT)

/kaggle/working
Cloning into 'llama.cpp'...
remote: Enumerating objects: 3768, done.
remote: Counting objects: 100% (3768/3768), done.
remote: Compressing objects: 100% (3065/3065), done.
remote: Total 3768 (delta 657), reused 2694 (delta 619), pack-reused 0 (from 0)
Receiving objects: 100% (3768/3768), 35.33 MiB | 20.92 MiB/s, done.
Resolving deltas: 100% (657/657), done.
/kaggle/working/llama.cpp
Building commit: 9b05354


In [3]:
# Opsional tapi disarankan: reset file ke versi asli dari git supaya bersih dari patch2 gagal sebelumnya
%cd /kaggle/working/llama.cpp
!git checkout -- ggml/src/ggml-cuda/CMakeLists.txt

path = "/kaggle/working/llama.cpp/ggml/src/ggml-cuda/CMakeLists.txt"

with open(path, "r") as f:
    content = f.read()

marker = "target_link_libraries(ggml-cuda PRIVATE CUDA::cuda_driver)"

patch = '''if (NOT TARGET CUDA::cuda_driver)
        add_library(CUDA::cuda_driver SHARED IMPORTED GLOBAL)
        set_target_properties(CUDA::cuda_driver PROPERTIES
            IMPORTED_LOCATION "/usr/local/nvidia/lib64/libcuda.so"
            IMPORTED_NO_SONAME TRUE)
        message(STATUS "MANUAL PATCH: CUDA::cuda_driver created -> /usr/local/nvidia/lib64/libcuda.so")
    endif()
    '''

assert content.count(marker) == 1, f"marker ditemukan {content.count(marker)} kali, harus tepat 1"

idx = content.index(marker)
new_content = content[:idx] + patch + content[idx:]
with open(path, "w") as f:
    f.write(new_content)
print("Patch berhasil disisipkan tepat sebelum baris target_link_libraries(ggml-cuda PRIVATE CUDA::cuda_driver)")

/kaggle/working/llama.cpp
Patch berhasil disisipkan tepat sebelum baris target_link_libraries(ggml-cuda PRIVATE CUDA::cuda_driver)


In [4]:
%cd /kaggle/working/llama.cpp
!rm -rf build
!cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF

/kaggle/working/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.1.0-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYS

In [ ]:
!cmake --build build --config Release -j$(nproc) --target llama-server llama-cli

[  0%] Building CXX object common/CMakeFiles/llama-common-base.dir/build-info.cpp.o
[  0%] Building CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o
[  0%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml.c.o
[  0%] Building CXX object tools/ui/CMakeFiles/llama-ui-embed.dir/embed.cpp.o
[  0%] Linking CXX static library libllama-common-base.a
[  0%] Built target llama-common-base
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml.cpp.o
[  1%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-alloc.c.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend.cpp.o
[  1%] Linking CXX executable llama-ui-embed
[  1%] Built target llama-ui-embed
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend-meta.cpp.o
[  2%] Provisioning UI assets
-- UI: running npm ci
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-opt.cpp.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-thr

## 2. Kumpulkan hasil build ke folder output

Kita ambil binary + shared library (`.so`) yang dibutuhkan supaya `llama-server` bisa
langsung dijalankan di notebook lain tanpa `LD_LIBRARY_PATH` tambahan (semua file diletakkan
satu folder).


In [ ]:
import os, shutil, glob

OUT_DIR = "/kaggle/working/llama-cpp-cuda-sm75-build"
os.makedirs(OUT_DIR, exist_ok=True)

BUILD_BIN_DIR = "/kaggle/working/llama.cpp/build/bin"

# Salin semua binary + .so hasil build (llama-server, llama-cli, libllama.so, libggml*.so, dst)
copied = []
for pattern in ("llama-server", "llama-cli", "*.so", "*.so.*"):
    for path in glob.glob(os.path.join(BUILD_BIN_DIR, pattern)):
        dest = os.path.join(OUT_DIR, os.path.basename(path))
        shutil.copy2(path, dest)
        copied.append(dest)

for f in sorted(copied):
    print(f)

# Simpan info commit supaya tahu versi llama.cpp yang dipakai
with open(os.path.join(OUT_DIR, "BUILD_INFO.txt"), "w") as f:
    f.write(f"llama.cpp commit: {LLAMA_CPP_COMMIT}\n")
    f.write("CUDA arch: sm75 (Tesla T4)\n")

print("\nTotal file:", len(copied))


## 3. Verifikasi binary bisa jalan (smoke test)

In [ ]:
import os

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = OUT_DIR + ":" + env.get("LD_LIBRARY_PATH", "")

!chmod +x {OUT_DIR}/llama-server {OUT_DIR}/llama-cli
!LD_LIBRARY_PATH={OUT_DIR} {OUT_DIR}/llama-server --version


## 4. Buat Kaggle Dataset dari hasil build

**Cara manual (disarankan):**
1. Klik **Save Version** (centang "Save & Run All").
2. Buka tab **Output** → **New Dataset**.
3. Nama, misal `llama-cpp-cuda-sm75-build`, lalu **Create**.
4. Nanti di notebook deploy: **Add Input → Datasets**, filenya ada di
   `/kaggle/input/llama-cpp-cuda-sm75-build/llama-cpp-cuda-sm75-build/llama-server`.

**Cara otomatis via Kaggle API + Kaggle Secrets (tanpa file `kaggle.json`):**
1. **Add-ons → Secrets** → tambahkan `KAGGLE_USERNAME` dan `KAGGLE_KEY` (dari
   Account Settings → Create New API Token di Kaggle), lalu **Attach** ke notebook ini.
2. Jalankan cell di bawah.


In [ ]:
# OPSIONAL — otomatisasi lewat Kaggle API, pakai Kaggle Secrets (bukan file kaggle.json).
# Skip cell ini kalau kamu pakai cara manual (Save Version -> New Dataset).

import os, json, subprocess

DATASET_SLUG = "llama-cpp-cuda-sm75-build"
DATASET_TITLE = "llama.cpp CUDA build (sm75 / Tesla T4)"

from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")
    have_secrets = True
except Exception as e:
    have_secrets = False
    print(
        "Secret KAGGLE_USERNAME / KAGGLE_KEY belum di-set atau belum di-attach. "
        "Buka Add-ons -> Secrets untuk menambahkan & mengaktifkannya.\n"
        f"Detail error: {e}"
    )

if have_secrets:
    metadata = {
        "title": DATASET_TITLE,
        "id": f"{os.environ['KAGGLE_USERNAME']}/{DATASET_SLUG}",
        "licenses": [{"name": "MIT"}],
    }
    with open(os.path.join(OUT_DIR, "dataset-metadata.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    # "create" untuk dataset baru. Kalau dataset dengan slug ini sudah ada, ganti ke:
    # subprocess.run(["kaggle", "datasets", "version", "-p", OUT_DIR, "-m", "update build"], check=True)
    subprocess.run(["kaggle", "datasets", "create", "-p", OUT_DIR, "-r", "zip"], check=True)
    print("Dataset dibuat / diupdate:", metadata["id"])
else:
    print("Pakai cara manual (Save Version -> New Dataset) saja.")


## Catatan

- **Kapan perlu rebuild & update dataset ini?** Kalau kamu mau upgrade versi `llama.cpp`
  (misal ada arsitektur model baru yang belum didukung versi lama), atau kalau environment
  CUDA di image Kaggle berubah versi major. Selain itu, dataset ini reusable terus tanpa
  compile ulang.
- **Kenapa salin `.so` juga, bukan cuma binary?** `llama-server`/`llama-cli` di build shared
  (bukan static), jadi butuh `libllama.so`, `libggml*.so`, dll ada di folder yang sama /
  `LD_LIBRARY_PATH` supaya bisa jalan di notebook lain.
- **Kalau GPU Kaggle nanti bukan T4** (compute capability beda dari 7.5), binary ini tidak
  akan jalan optimal / bisa gagal load CUDA kernel — perlu rebuild dengan
  `CMAKE_CUDA_ARCHITECTURES` yang sesuai.
